# Week 94, Day 1: Reading large codebases

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('minirepo.py', 'w', encoding='utf-8') as f:
    f.write('REPO = {\n"README.md": \'\'\'# pathkit\n\nSmall path-finding helpers: a graph, shortest routes, CSV loading.\n\n    python -m pathkit.cli map.csv a c\n\'\'\',\n"pathkit/__init__.py": \'\'\'"""A small library of path-finding helpers."""\nfrom pathkit.graph import Graph\nfrom pathkit.search import shortest\n\n__all__ = ["Graph", "shortest"]\n\'\'\',\n"pathkit/graph.py": \'\'\'class Graph:\n    """An undirected graph with a cost on each edge."""\n\n    def __init__(self):\n        self.edges = {}\n\n    def add(self, a, b, cost):\n        self.edges.setdefault(a, {})[b] = cost\n        self.edges.setdefault(b, {})[a] = cost\n\n    def neighbours(self, node):\n        return sorted(self.edges.get(node, {}).items())\n\n\ndef from_pairs(pairs):\n    graph = Graph()\n    for a, b, cost in pairs:\n        graph.add(a, b, cost)\n    return graph\n\'\'\',\n"pathkit/search.py": \'\'\'import heapq\n\nfrom pathkit.graph import Graph\n\n\ndef shortest(graph, start, end):\n    heap, seen = [(0, start, [start])], set()\n    while heap:\n        cost, node, path = heapq.heappop(heap)\n        if node == end:\n            return cost, path\n        if node in seen:\n            continue\n        seen.add(node)\n        for nxt, step in graph.neighbours(node):\n            heapq.heappush(heap, (cost + step, nxt, path + [nxt]))\n    return None\n\n\ndef reachable(graph, start):\n    found, todo = {start}, [start]\n    while todo:\n        for nxt, _ in graph.neighbours(todo.pop()):\n            if nxt not in found:\n                found.add(nxt)\n                todo.append(nxt)\n    return found\n\'\'\',\n"pathkit/io.py": \'\'\'import csv\n\nfrom pathkit.graph import from_pairs\n\n\ndef load_csv(path):\n    with open(path, newline="") as f:\n        rows = [(a, b, float(cost)) for a, b, cost in csv.reader(f)]\n    return from_pairs(rows)\n\'\'\',\n"pathkit/cli.py": \'\'\'import sys\n\nfrom pathkit.io import load_csv\nfrom pathkit.search import shortest\n\n\ndef main(argv=None):\n    argv = sys.argv[1:] if argv is None else argv\n    graph = load_csv(argv[0])\n    result = shortest(graph, argv[1], argv[2])\n    if result is None:\n        print("no route")\n        return 1\n    cost, path = result\n    print(" to ".join(path), cost)\n    return 0\n\'\'\',\n"tests/test_graph.py": \'\'\'from pathkit.graph import Graph, from_pairs\n\n\ndef test_add_is_symmetric():\n    g = Graph()\n    g.add("a", "b", 2)\n    assert g.neighbours("b") == [("a", 2)]\n\n\ndef test_from_pairs():\n    g = from_pairs([("a", "b", 1), ("b", "c", 2)])\n    assert g.neighbours("b") == [("a", 1), ("c", 2)]\n\'\'\',\n"tests/test_search.py": \'\'\'from pathkit.graph import from_pairs\nfrom pathkit.search import shortest\n\n\ndef test_shortest():\n    g = from_pairs([("a", "b", 1), ("b", "c", 2), ("a", "c", 5)])\n    assert shortest(g, "a", "c") == (3, ["a", "b", "c"])\n\'\'\',\n}\n')

**Predict before you run:** what will this print?



In [ ]:
from minirepo import REPO

print(len(REPO), sorted(p for p in REPO if p.startswith("tests/")))
print(sum(len(src.splitlines()) for src in REPO.values()))

> **Key idea.** Read from the outside in: README, entry points, tests, then one question at a time, with notes. Search with `git grep -n`. Index definitions (functions, classes, methods) with the syntax tree. Map internal imports: much-imported modules are the core, unimported ones are entry points or dead code, and a topological order (alphabetical ties) gives a reading order; a cycle means none exists. Find callers by name, as candidates. Note untested modules and uncalled functions.

### Exercise: Searching

Write `grep(repo, pattern)`: return `"PATH:LINE: TEXT"` for every line (stripped of surrounding whitespace) matching the regular expression `pattern`, with files in sorted path order and line numbers from 1.

In [ ]:
import re

def grep(repo, pattern):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\ndef test_grep():\n    \"every use of neighbours\"\n    assert grep(REPO, r\"neighbours\\(\") == [\n        \"pathkit/graph.py:11: def neighbours(self, node):\",\n        \"pathkit/search.py:15: for nxt, step in graph.neighbours(node):\",\n        \"pathkit/search.py:23: for nxt, _ in graph.neighbours(todo.pop()):\",\n        'tests/test_graph.py:7: assert g.neighbours(\"b\") == [(\"a\", 2)]',\n        'tests/test_graph.py:12: assert g.neighbours(\"b\") == [(\"a\", 1), (\"c\", 2)]',\n    ]\n    assert grep(REPO, r\"^import \") == [\"pathkit/cli.py:1: import sys\", \"pathkit/io.py:1: import csv\", \"pathkit/search.py:1: import heapq\"]\n    assert grep(REPO, \"no such thing\") == []")

### Exercise: Where is it defined?

Write `definitions(repo)`: for every `.py` file (in sorted path order), index each top-level function and class, and each method of those classes (as `Class.method`), mapping the name to a list of `"PATH:LINE"` (a name defined in several places has several entries).

In [ ]:
import ast

def definitions(repo):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\ndef test_index():\n    \"go to definition\"\n    index = definitions(REPO)\n    assert index[\"Graph\"] == [\"pathkit/graph.py:1\"] and index[\"Graph.neighbours\"] == [\"pathkit/graph.py:11\"]\n    assert index[\"from_pairs\"] == [\"pathkit/graph.py:15\"] and index[\"shortest\"] == [\"pathkit/search.py:6\"]\n    assert index[\"main\"] == [\"pathkit/cli.py:7\"] and index[\"test_shortest\"] == [\"tests/test_search.py:5\"]\n    assert len(index) == 12\n\ndef test_twice():\n    \"the same name in two files\"\n    repo = {\"b.py\": \"def run():\\n    pass\\n\", \"a.py\": \"\\n\\ndef run():\\n    pass\\n\", \"notes.md\": \"def run(): not code\"}\n    assert definitions(repo) == {\"run\": [\"a.py:3\", \"b.py:1\"]}")

### Exercise: The import graph

Using the given `module_name` (a path to a dotted module name, with `__init__.py` naming its package) and `imports` (the modules a source imports), write `import_graph(repo)`: map each module in the repository (from its `.py` files) to a **sorted list** of the repository's own modules it imports, ignoring everything else, with the modules in sorted order.

In [ ]:
import ast


def module_name(path):
    parts = path[:-3].split("/")
    if parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def imports(source):
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            found.add(node.module)
    return found

def import_graph(repo):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\ndef test_graph():\n    \"pathkit's shape\"\n    assert import_graph(REPO) == {\n        \"pathkit\": [\"pathkit.graph\", \"pathkit.search\"],\n        \"pathkit.cli\": [\"pathkit.io\", \"pathkit.search\"],\n        \"pathkit.graph\": [],\n        \"pathkit.io\": [\"pathkit.graph\"],\n        \"pathkit.search\": [\"pathkit.graph\"],\n        \"tests.test_graph\": [\"pathkit.graph\"],\n        \"tests.test_search\": [\"pathkit.graph\", \"pathkit.search\"],\n    }\n    assert list(import_graph(REPO)) == sorted(import_graph(REPO))")

### Exercise: A reading order

Write `reading_order(graph)`: `graph` maps each module to the modules it imports. Return the modules so that each comes after everything it imports, always taking the alphabetically first module that's ready (use `heapq`). Raise `ValueError("import cycle among: A, B")` naming, sorted, the modules that could never be placed.

In [ ]:
import heapq

def reading_order(graph):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GRAPH = {\"pathkit\": [\"pathkit.graph\", \"pathkit.search\"], \"pathkit.cli\": [\"pathkit.io\", \"pathkit.search\"], \"pathkit.graph\": [],\n         \"pathkit.io\": [\"pathkit.graph\"], \"pathkit.search\": [\"pathkit.graph\"], \"tests.test_graph\": [\"pathkit.graph\"],\n         \"tests.test_search\": [\"pathkit.graph\", \"pathkit.search\"]}\n\ndef test_order():\n    \"dependencies first\"\n    assert reading_order(GRAPH) == [\"pathkit.graph\", \"pathkit.io\", \"pathkit.search\", \"pathkit\", \"pathkit.cli\",\n                                    \"tests.test_graph\", \"tests.test_search\"]\n\ndef test_cycle():\n    \"no order exists\"\n    try:\n        reading_order({\"a\": [], \"b\": [\"c\"], \"c\": [\"b\"], \"d\": [\"c\"]})\n    except ValueError as e:\n        assert str(e) == \"import cycle among: b, c, d\"\n    else:\n        raise AssertionError(\"cycle\")")

### Exercise: Who calls it?

Write `callers(repo, name)`: for each `.py` file in sorted order, look at its top-level functions and the methods of its classes (in order), and return `"PATH:FUNCTION"` (methods as `Class.method`) for each that contains a call to `name`, either as a plain name (`name(...)`) or as an attribute (`something.name(...)`). List each function once.

In [ ]:
import ast

def callers(repo, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\ndef test_callers():\n    \"how it's really used\"\n    assert callers(REPO, \"from_pairs\") == [\"pathkit/io.py:load_csv\", \"tests/test_graph.py:test_from_pairs\", \"tests/test_search.py:test_shortest\"]\n    assert callers(REPO, \"neighbours\") == [\"pathkit/search.py:shortest\", \"pathkit/search.py:reachable\",\n                                           \"tests/test_graph.py:test_add_is_symmetric\", \"tests/test_graph.py:test_from_pairs\"]\n    # set.add matches too: callers by name are candidates, not facts\n    assert callers(REPO, \"add\") == [\"pathkit/graph.py:from_pairs\", \"pathkit/search.py:shortest\", \"pathkit/search.py:reachable\",\n                                    \"tests/test_graph.py:test_add_is_symmetric\"]\n    assert callers(REPO, \"reachable\") == []")

### Bug hunt: A package called __init__

`module_name` turns `pathkit/__init__.py` into `pathkit.__init__`, but Python imports that file as `pathkit`, so imports of the package are never matched. It prints `['pathkit.__init__', 'pathkit.graph']`; fix it so it prints `['pathkit', 'pathkit.graph']`.

In [ ]:
def module_name(path):
    return path[:-3].replace("/", ".")

print([module_name(p) for p in ["pathkit/__init__.py", "pathkit/graph.py"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the package's own name\"\n    expect_output(\"['pathkit', 'pathkit.graph']\")")

### Bug hunt: Imports hidden in functions

This only looks at the top level of the module, so it misses the import inside `main` and prints `['sys']`. Walk the whole tree so it prints `['pathkit.io', 'sys']`.

In [ ]:
import ast

SOURCE = "import sys\n\ndef main():\n    from pathkit.io import load_csv\n    return load_csv(sys.argv[1])\n"

def imports(source):
    found = set()
    for node in ast.parse(source).body:
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module:
            found.add(node.module)
    return sorted(found)

print(imports(SOURCE))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ast.walk\"\n    expect_output(\"['pathkit.io', 'sys']\")")

### Puzzle: Dependencies first

Arrange the loop of `reading_order`: take the alphabetically first ready module, then release the modules that were waiting only for it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # module_name("pathkit/__init__.py")
q = ___   # the module in pathkit that the most other modules import
r = ___   # the first module in pathkit's reading order
s = ___   # True or False: callers(REPO, "reachable") finds anything
t = ___   # the git command that searches tracked files, with line numbers (as you'd type it, without the pattern)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"pathkit\" and q == \"pathkit.graph\" and r == \"pathkit.graph\", \"p, q and r\"\n    assert s is False and t == \"git grep -n\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an orientation report

Write `orientation(repo)` using the given helpers. Test modules are those whose name starts with `tests.` or whose last part starts with `test_`; the others are source modules. Return these lines:

1. `"files: F, P Python, L lines"` (all files, the `.py` ones, and the lines in all files);
2. `"largest: PATH (N), PATH (N), PATH (N)"`: the three `.py` files with most lines (ties by path);
3. `"most imported: MODULE (K), ..."`: the three modules imported by the most other modules (ties by name, leaving out modules nobody imports);
4. `"entry points: ..."`: source modules nobody imports, sorted;
5. `"untested: ..."`: source modules no test module imports, sorted;
6. `"not called in the repo: ..."`: top-level functions in source modules (not starting with `_`) that no function in the repository calls by name, sorted;
7. `"start with: A, B, C"`: the first three modules of the reading order.

Write `none` for an empty list.

In [ ]:
import ast


def module_name(path):
    parts = path[:-3].split("/")
    if parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def imports(source):
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            found.add(node.module)
    return found


def orientation(repo):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\ndef test_pathkit():\n    \"first day on pathkit\"\n    assert orientation(REPO) == [\n        \"files: 8, 7 Python, 100 lines\",\n        \"largest: pathkit/search.py (27), pathkit/graph.py (19), pathkit/cli.py (16)\",\n        \"most imported: pathkit.graph (5), pathkit.search (3), pathkit.io (1)\",\n        \"entry points: pathkit, pathkit.cli\",\n        \"untested: pathkit, pathkit.cli, pathkit.io\",\n        \"not called in the repo: main, reachable\",\n        \"start with: pathkit.graph, pathkit.io, pathkit.search\",\n    ]\n\ndef test_tiny():\n    \"one module\"\n    assert orientation({\"tool.py\": \"def _helper():\\n    pass\\n\"}) == [\n        \"files: 1, 1 Python, 2 lines\", \"largest: tool.py (2)\", \"most imported: none\", \"entry points: tool\", \"untested: tool\",\n        \"not called in the repo: none\", \"start with: tool\"]")

## Warm-up

`minirepo.py` holds a whole small project, `pathkit`, as a dictionary from file path to file contents. A hundred lines is small enough to read in full, which is the point: today you'll build tools on something you can check by eye, and then use the real versions (`git grep`, your editor's "go to definition") on projects of a hundred thousand lines.

## Strategies for reading

Start from the **outside**: the README (what is it for?), how it's run (a command, a library's public names in `__init__.py`), and the **tests**, which are examples of use written by the people who know the code best. Then go **inside** with a question, not a plan to read everything: "where does a route get its cost?" Follow that one thread, writing down what you learn (a page of notes beats a perfect memory). Expect to understand a small part well and the rest vaguely. That's normal, and enough to start contributing.

## Search and definitions

**Search** is the most used tool: `git grep -n "neighbours("` lists every matching line with its file and line number, much faster than opening files. Search for a name to see where it's used; search for an error message to find where it's raised.

A **definitions index** answers "where is this defined?" without guessing from file names. The syntax tree knows: every `def` and `class` at the top of a module, and every method inside a class, with its line number. Editors build exactly this index for "go to definition".

## Imports and reading order

Imports show the **shape** of a project: which modules depend on which. Draw the graph (module to the modules it imports, ignoring the standard library), and three things stand out. Modules many others import are the **core** (in `pathkit`, `graph`): read them first. Modules nothing imports are **entry points** (the command line, the package's `__init__`) or dead code. And a **reading order** that puts every module after the ones it depends on (a topological order, with ties broken alphabetically to make it repeatable) lets you read without constantly jumping ahead. If there's a cycle, there's no such order: two modules import each other, and that's worth knowing too.

## Calls and tests

"Who calls this?" tells you how a function is really used, and so what a change to it might break. The syntax tree finds every call by name, both `from_pairs(...)` and `graph.neighbours(...)`. Search by name can't know types, so treat the results as candidates: in `pathkit`, the callers of `add` include two functions that call `set.add`, nothing to do with `Graph.add`. That's true of every tool this week.

Two more questions are worth asking on the first day: which modules no test imports (changes there are risky), and which functions nothing calls (perhaps called from outside, perhaps dead).

## Practice

## Quiz

## Daily challenge

## Your turn on a real repository

Pick a Python project you use (a library such as `requests`, `rich` or `attrs`, or the standard library's own `Lib/json` in CPython) and clone it on your own computer.

1. **Read the outside**: the README, `CONTRIBUTING`, how it's installed and run, and one test file.
2. **Run your tools**: adapt today's functions to read a folder (with `pathlib.Path.rglob("*.py")`) and produce its orientation report.
3. **Ask one question** ("how does it decide X?") and answer it with search, definitions and callers, writing a page of notes.

## Before you go

Tomorrow is about the people around the code: how open-source projects are run, licensed, and kept alive, and what that means for a newcomer.